# Fuzzy RDD: When the Cutoff Doesn't Perfectly Determine Treatment

In **sharp RDD**, crossing the cutoff deterministically assigns treatment: everyone above gets treated, everyone below doesn't. But many real-world settings are messier than this.

In **fuzzy RDD**, crossing the cutoff *increases the probability* of treatment but doesn't guarantee it. Some people above the cutoff don't take up treatment; some below the cutoff manage to get it anyway.

This is exactly the setup in **Mo & Conn (2018)**: Teach For America corps members are selected based on a test score cutoff, but scoring above the cutoff doesn't guarantee placement (some decline, some aren't matched to schools), and some below-cutoff applicants end up teaching through other routes.

The solution: treat the cutoff as an **instrument** and use **2SLS/IV estimation**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(42)

# Plotting style (match rdd-concepts.ipynb)
plt.rcParams.update({
    'figure.figsize': (9, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Simulating Fuzzy RDD Data

We simulate a TFA-like setting:
- **Running variable** $X$: selection score, centered at the cutoff
- **Cutoff** at $X = 0$
- **Above the cutoff**: 60% chance of actually teaching (some decline, aren't placed, etc.)
- **Below the cutoff**: 10% chance of teaching (some self-select in through other routes)

The key difference from sharp RDD: the instrument $Z_i = \mathbf{1}(X_i \geq 0)$ does not equal the treatment $D_i$. Instead, $Z$ shifts the *probability* of $D$.

In [ ]:
n = 2000
cutoff = 0

# Running variable (selection score, centered at cutoff)
X = np.random.normal(0, 1, n)

# Fuzzy treatment: above cutoff INCREASES probability, doesn't guarantee it
# Below cutoff: 10% chance of treatment (some people self-select in)
# Above cutoff: 60% chance (admission doesn't guarantee participation)
prob_treat = np.where(X >= cutoff, 0.6, 0.1)
D = np.random.binomial(1, prob_treat)

# Instrument: above cutoff indicator
Z = (X >= cutoff).astype(int)

# Outcome: Y = 2 + 3*D + 1.5*X + noise
# True treatment effect = 3
true_effect = 3
Y = 2 + true_effect * D + 1.5 * X + np.random.normal(0, 1, n)

df = pd.DataFrame({'X': X, 'D': D, 'Z': Z, 'Y': Y})

print(f"N = {n}")
print(f"P(D=1 | X >= cutoff) = {D[X >= cutoff].mean():.3f}")
print(f"P(D=1 | X < cutoff)  = {D[X < cutoff].mean():.3f}")
print(f"First stage jump     = {D[X >= cutoff].mean() - D[X < cutoff].mean():.3f}")
print(f"\nIf this were sharp RDD, the first stage jump would be 1.000")
print(f"The gap between 1.0 and {D[X >= cutoff].mean() - D[X < cutoff].mean():.3f} is the 'fuzziness'")

## 2. Visualizing the Fuzzy First Stage

In sharp RDD, a plot of treatment status vs. the running variable shows a perfect step function at the cutoff. In fuzzy RDD, we see a *jump* in the probability of treatment, but it goes from (say) 10% to 60%, not from 0% to 100%.

This is the **first stage** of the IV setup: $Z$ (above cutoff) predicts $D$ (treatment), but imperfectly.

In [ ]:
# Binned scatter: probability of treatment vs running variable
n_bins = 30
bin_edges = np.linspace(X.min(), X.max(), n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_treat_prob = np.zeros(n_bins)
bin_counts = np.zeros(n_bins)

for i in range(n_bins):
    mask = (X >= bin_edges[i]) & (X < bin_edges[i+1])
    if mask.sum() > 0:
        bin_treat_prob[i] = D[mask].mean()
        bin_counts[i] = mask.sum()

fig, ax = plt.subplots(figsize=(10, 6))

below = bin_centers < cutoff
above = bin_centers >= cutoff
ax.scatter(bin_centers[below], bin_treat_prob[below], color='steelblue', s=50, zorder=5, label='Below cutoff')
ax.scatter(bin_centers[above], bin_treat_prob[above], color='darkorange', s=50, zorder=5, label='Above cutoff')

# Horizontal reference lines
ax.axhline(y=0.1, color='steelblue', linestyle=':', alpha=0.5, label='P(D=1) below = 0.10')
ax.axhline(y=0.6, color='darkorange', linestyle=':', alpha=0.5, label='P(D=1) above = 0.60')
ax.axvline(x=cutoff, color='gray', linestyle='--', linewidth=1, alpha=0.7)

# Annotate the jump
ax.annotate('', xy=(0.05, 0.6), xytext=(0.05, 0.1),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.15, 0.35, 'First stage\njump = 0.50', fontsize=12, color='red', va='center')

ax.set_xlabel('Running Variable (Selection Score)', fontsize=12)
ax.set_ylabel('P(Treatment = 1)', fontsize=12)
ax.set_title('Fuzzy First Stage: Treatment Probability Jumps at Cutoff\n(but neither side is 0% or 100%)', fontsize=13)
ax.legend(loc='upper left')
ax.set_ylim(-0.05, 1.0)
plt.tight_layout()
plt.show()

print("Compare to sharp RDD: there, P(D=1) jumps from 0.0 to 1.0 at the cutoff.")
print("Here, it jumps from ~0.10 to ~0.60. This is fuzzy.")

## 3. The Reduced Form (Intent-to-Treat)

The **reduced form** is the relationship between the instrument $Z$ (above cutoff) and the outcome $Y$, ignoring actual treatment status. This gives the **intent-to-treat (ITT)** effect: the effect of being *assigned* to treatment (scoring above the cutoff), regardless of whether you actually take it up.

In Mo & Conn's setting: the reduced form answers "what is the effect of scoring above the TFA cutoff on student outcomes?" This is interesting on its own, but it underestimates the effect of actually having a TFA teacher (since not everyone above the cutoff gets one).

In [ ]:
# Reduced form: outcome vs running variable (ignoring treatment status)
fig, ax = plt.subplots(figsize=(10, 6))

# Binned scatter of outcomes
bin_outcome = np.zeros(n_bins)
for i in range(n_bins):
    mask = (X >= bin_edges[i]) & (X < bin_edges[i+1])
    if mask.sum() > 0:
        bin_outcome[i] = Y[mask].mean()

ax.scatter(bin_centers[below], bin_outcome[below], color='steelblue', s=50, zorder=5)
ax.scatter(bin_centers[above], bin_outcome[above], color='darkorange', s=50, zorder=5)

# Fit lines on each side
for mask, color in [(X < cutoff, 'steelblue'), (X >= cutoff, 'darkorange')]:
    slope, intercept, _, _, _ = stats.linregress(X[mask], Y[mask])
    x_line = np.linspace(X[mask].min(), X[mask].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color=color, linewidth=2.5)

ax.axvline(x=cutoff, color='gray', linestyle='--', linewidth=1, alpha=0.7)

# Compute and annotate the reduced form jump
slope_b, int_b, _, _, _ = stats.linregress(X[X < cutoff], Y[X < cutoff])
slope_a, int_a, _, _, _ = stats.linregress(X[X >= cutoff], Y[X >= cutoff])
y_below_cf = int_b + slope_b * cutoff
y_above_cf = int_a + slope_a * cutoff
rf_jump = y_above_cf - y_below_cf

ax.annotate('', xy=(0.05, y_above_cf), xytext=(0.05, y_below_cf),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.15, (y_above_cf + y_below_cf) / 2, f'Reduced form\njump = {rf_jump:.2f}',
        fontsize=12, color='red', va='center')

ax.set_xlabel('Running Variable (Selection Score)', fontsize=12)
ax.set_ylabel('Outcome (Y)', fontsize=12)
ax.set_title('Reduced Form: Outcome vs. Running Variable\n(This is the ITT effect)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Reduced form jump (ITT): {rf_jump:.3f}")
print(f"True treatment effect:   {true_effect}")
print(f"\nThe ITT ({rf_jump:.2f}) is smaller than the true effect ({true_effect})")
print(f"because only ~50% of people above the cutoff actually get treated.")

## 4. New Concept: Instrumental Variables (IV)

Before we can estimate the fuzzy RDD treatment effect, we need a new tool. In sharp RDD, the jump at the cutoff *is* the treatment effect. But in fuzzy RDD, the jump at the cutoff is attenuated because not everyone above the cutoff gets treated.

We need **instrumental variables (IV)**: a method for estimating causal effects when treatment is not perfectly assigned.

### The core idea

An **instrument** $Z$ is a variable that:

1. **Affects treatment** (relevance): $Z$ changes the probability of getting treated
2. **Only affects the outcome through treatment** (exclusion restriction): $Z$ has no direct effect on $Y$ except via $D$
3. **Is as good as randomly assigned** (independence): $Z$ is unrelated to confounders

**Everyday analogy**: Suppose you want to know whether exercise improves health, but people who exercise are already healthier (selection bias). An instrument could be *proximity to a gym*: living near a gym *encourages* exercise, but living near a gym does not directly improve your health (only through exercise). Compare health outcomes of people near vs. far from gyms, and scale by how much proximity changes exercise behavior.

In our RDD context, the instrument is $Z_i = \mathbf{1}(X_i \geq 0)$: scoring above the cutoff. It satisfies all three conditions:
1. Being above the cutoff increases TFA participation (the first stage)
2. Scoring above the cutoff only affects attitudes *through* TFA participation
3. Near the cutoff, being above or below is as good as random

In [ ]:
# Visualize the IV logic: Z -> D -> Y
# The instrument (above cutoff) shifts treatment, treatment shifts outcome

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: First stage (Z -> D)
ax = axes[0]
z_vals = [0, 1]
d_means = [D[Z == 0].mean(), D[Z == 1].mean()]
ax.bar(z_vals, d_means, color=['steelblue', 'darkorange'], width=0.4, edgecolor='black')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Z = 0\n(Below cutoff)', 'Z = 1\n(Above cutoff)'])
ax.set_ylabel('P(Treatment = 1)')
ax.set_title('First Stage: Z → D\n(Instrument shifts treatment probability)', fontsize=12)
ax.set_ylim(0, 0.8)
for i, v in enumerate(d_means):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=14, fontweight='bold')
ax.annotate('', xy=(1, d_means[1] - 0.02), xytext=(0, d_means[0] + 0.02),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(0.5, 0.25, f'First stage\n= {d_means[1] - d_means[0]:.2f}', ha='center',
        fontsize=12, color='red', fontweight='bold')

# Right panel: Reduced form (Z -> Y, the ITT)
ax = axes[1]
y_means = [Y[Z == 0].mean(), Y[Z == 1].mean()]
ax.bar(z_vals, y_means, color=['steelblue', 'darkorange'], width=0.4, edgecolor='black')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Z = 0\n(Below cutoff)', 'Z = 1\n(Above cutoff)'])
ax.set_ylabel('Mean Outcome (Y)')
ax.set_title('Reduced Form: Z → Y\n(Effect of instrument on outcome)', fontsize=12)
for i, v in enumerate(y_means):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=14, fontweight='bold')
ax.annotate('', xy=(1, y_means[1] - 0.05), xytext=(0, y_means[0] + 0.05),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
rf = y_means[1] - y_means[0]
ax.text(0.5, (y_means[0] + y_means[1]) / 2, f'Reduced form\n= {rf:.2f}', ha='center',
        fontsize=12, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

# Show the Wald calculation
fs = d_means[1] - d_means[0]
print(f"The IV/Wald estimator:")
print(f"  τ_IV = reduced form / first stage")
print(f"       = {rf:.3f} / {fs:.3f}")
print(f"       = {rf/fs:.3f}")
print(f"  True effect = {true_effect}")
print(f"\nIntuition: the reduced form ({rf:.2f}) is the effect of the *encouragement*.")
print(f"The first stage ({fs:.2f}) is how much the encouragement shifts treatment.")
print(f"Dividing scales up to get the effect of treatment *itself*.")

## 5. Applying IV: The Wald Estimator and 2SLS

Now that we understand IV, we can estimate the fuzzy RDD treatment effect. Two equivalent methods:

**Wald estimator** (for binary instruments):
$$\hat{\tau}_{\text{IV}} = \frac{E[Y | Z=1] - E[Y | Z=0]}{E[D | Z=1] - E[D | Z=0]} = \frac{\text{reduced form}}{\text{first stage}}$$

**Two-stage least squares (2SLS)**:
- **Stage 1**: Regress $D$ on $Z$ (and controls like $X$). Get predicted $\hat{D}$.
- **Stage 2**: Regress $Y$ on $\hat{D}$ (and controls). The coefficient on $\hat{D}$ is the IV estimate.

Why 2SLS works: $\hat{D}$ contains only the variation in treatment coming from the instrument (the "clean" variation, free of selection bias). By using $\hat{D}$ instead of $D$, we strip out self-selection.

In [ ]:
# === Method 1: Manual Wald estimator within a bandwidth ===
h = 0.5  # bandwidth
in_bw = np.abs(X) <= h

# First stage: E[D|Z=1] - E[D|Z=0] within bandwidth
first_stage = D[(X >= 0) & in_bw].mean() - D[(X < 0) & in_bw].mean()

# Reduced form: E[Y|Z=1] - E[Y|Z=0] within bandwidth
reduced_form = Y[(X >= 0) & in_bw].mean() - Y[(X < 0) & in_bw].mean()

wald_estimate = reduced_form / first_stage

print("=== Method 1: Wald Estimator (manual) ===")
print(f"Bandwidth: {h}")
print(f"Observations in bandwidth: {in_bw.sum()}")
print(f"First stage:    {first_stage:.3f}")
print(f"Reduced form:   {reduced_form:.3f}")
print(f"Wald estimate:  {wald_estimate:.3f}")
print(f"True effect:    {true_effect}")
print()

# === Method 2: 2SLS by hand ===
from numpy.linalg import inv

Xbw = X[in_bw]
Ybw = Y[in_bw]
Dbw = D[in_bw]
Zbw = Z[in_bw]

# First stage: D on Z and X
W = np.column_stack([np.ones(Zbw.shape[0]), Zbw, Xbw])
first_stage_coef = inv(W.T @ W) @ (W.T @ Dbw)
D_hat = W @ first_stage_coef

print(f"=== Method 2: 2SLS (manual matrix algebra) ===")
print(f"First stage: D = {first_stage_coef[0]:.3f} + {first_stage_coef[1]:.3f}*Z + {first_stage_coef[2]:.3f}*X")

# Second stage: Y on D_hat and X
W2 = np.column_stack([np.ones(D_hat.shape[0]), D_hat, Xbw])
second_stage_coef = inv(W2.T @ W2) @ (W2.T @ Ybw)

print(f"Second stage: Y = {second_stage_coef[0]:.3f} + {second_stage_coef[1]:.3f}*D_hat + {second_stage_coef[2]:.3f}*X")
print(f"2SLS estimate:  {second_stage_coef[1]:.3f}")
print(f"True effect:    {true_effect}")
print()

# === Method 3: Naive OLS (biased) for comparison ===
df_bw = df[in_bw].copy()
df_bw['XD'] = df_bw['X'] * df_bw['D']
ols_result = smf.ols('Y ~ D + X + XD', data=df_bw).fit()

print("=== Method 3: Naive OLS (for comparison) ===")
print(f"OLS estimate:   {ols_result.params['D']:.3f}")
print(f"True effect:    {true_effect}")
print(f"\nOLS is close here because our simulation has no confounding beyond X.")
print(f"In practice, OLS can be badly biased when unobservables affect both D and Y.")

## 6. Sharp vs. Fuzzy: Side-by-Side Comparison

To build intuition, let's generate both sharp and fuzzy RDD data from the same underlying population and compare. The treatment effect is the same ($\tau = 3$) in both cases; the difference is in how treatment is assigned.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Use the same X and noise for fair comparison
np.random.seed(123)
n_comp = 1000
X_comp = np.random.uniform(-1, 1, n_comp)
eps = np.random.normal(0, 0.5, n_comp)

# --- Sharp RDD ---
D_sharp = (X_comp >= 0).astype(int)
Y_sharp = 2 + true_effect * D_sharp + 1.5 * X_comp + eps

ax = axes[0]
ctrl = X_comp < 0
trt = X_comp >= 0
ax.scatter(X_comp[ctrl], Y_sharp[ctrl], alpha=0.3, s=15, color='steelblue')
ax.scatter(X_comp[trt], Y_sharp[trt], alpha=0.3, s=15, color='darkorange')

for mask, color in [(ctrl, 'steelblue'), (trt, 'darkorange')]:
    slope, intercept, _, _, _ = stats.linregress(X_comp[mask], Y_sharp[mask])
    x_line = np.linspace(X_comp[mask].min(), X_comp[mask].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color=color, linewidth=2.5)

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
ax.set_title('Sharp RDD\nD = 1(X >= 0)', fontsize=13)
ax.set_xlabel('Running Variable')
ax.set_ylabel('Outcome')
ax.set_ylim(-1, 8)

# --- Fuzzy RDD ---
prob_comp = np.where(X_comp >= 0, 0.6, 0.1)
D_fuzzy = np.random.binomial(1, prob_comp)
Y_fuzzy = 2 + true_effect * D_fuzzy + 1.5 * X_comp + eps

ax = axes[1]
# Color by actual treatment status (not cutoff side)
untreated = D_fuzzy == 0
treated = D_fuzzy == 1
ax.scatter(X_comp[untreated], Y_fuzzy[untreated], alpha=0.3, s=15, color='steelblue', label='Untreated')
ax.scatter(X_comp[treated], Y_fuzzy[treated], alpha=0.3, s=15, color='darkorange', label='Treated')

# Fit lines by cutoff side (not treatment status) for the reduced form
for mask, color in [(X_comp < 0, 'steelblue'), (X_comp >= 0, 'darkorange')]:
    slope, intercept, _, _, _ = stats.linregress(X_comp[mask], Y_fuzzy[mask])
    x_line = np.linspace(X_comp[mask].min(), X_comp[mask].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, color=color, linewidth=2.5, linestyle='--')

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
ax.set_title('Fuzzy RDD\nP(D=1) jumps from 0.1 to 0.6', fontsize=13)
ax.set_xlabel('Running Variable')
ax.set_ylim(-1, 8)
ax.legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

print("Left: Sharp RDD. Clean separation at the cutoff. Simple difference-in-means works.")
print("Right: Fuzzy RDD. Treated and untreated overlap on BOTH sides of the cutoff.")
print("       The reduced form jump (dashed lines) is attenuated relative to the true effect.")
print("       We need IV/2SLS to recover the treatment effect.")

## 7. Bandwidth Sensitivity

Just as in sharp RDD, the choice of bandwidth matters. Narrow bandwidths give less biased estimates (comparing truly similar units) but more variance (fewer observations). We sweep over a range of bandwidths and plot the resulting Wald/IV estimates.

In [ ]:
# Sweep over bandwidths for the fuzzy RDD Wald estimator
h_grid = np.arange(0.1, 2.01, 0.05)
results = []

for h in h_grid:
    in_bw = np.abs(X) <= h
    n_local = in_bw.sum()
    if n_local < 20:
        continue

    above_bw = (X >= 0) & in_bw
    below_bw = (X < 0) & in_bw

    if above_bw.sum() < 5 or below_bw.sum() < 5:
        continue

    fs = D[above_bw].mean() - D[below_bw].mean()
    rf = Y[above_bw].mean() - Y[below_bw].mean()

    if abs(fs) < 0.01:  # avoid division by near-zero
        continue

    wald = rf / fs

    # Bootstrap confidence interval
    np.random.seed(int(h * 100))
    boot_walds = []
    X_bw = X[in_bw]
    Y_bw = Y[in_bw]
    D_bw = D[in_bw]
    for _ in range(200):
        idx = np.random.choice(n_local, n_local, replace=True)
        Xb, Yb, Db = X_bw[idx], Y_bw[idx], D_bw[idx]
        fs_b = Db[Xb >= 0].mean() - Db[Xb < 0].mean() if (Xb >= 0).sum() > 0 and (Xb < 0).sum() > 0 else 0
        rf_b = Yb[Xb >= 0].mean() - Yb[Xb < 0].mean() if (Xb >= 0).sum() > 0 and (Xb < 0).sum() > 0 else 0
        if abs(fs_b) > 0.01:
            boot_walds.append(rf_b / fs_b)

    if len(boot_walds) > 10:
        ci_lo, ci_hi = np.percentile(boot_walds, [2.5, 97.5])
    else:
        ci_lo, ci_hi = np.nan, np.nan

    results.append({'h': h, 'wald': wald, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'n': n_local})

res = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(res['h'], res['wald'], 'o-', color='steelblue', markersize=4, label='Wald/IV estimate')
ax.fill_between(res['h'], res['ci_lo'], res['ci_hi'], alpha=0.2, color='steelblue', label='95% bootstrap CI')
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')

ax.set_xlabel('Bandwidth (h)', fontsize=12)
ax.set_ylabel('Estimated Treatment Effect', fontsize=12)
ax.set_title('Fuzzy RDD: Wald Estimate vs. Bandwidth', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print("The estimate is reasonably stable across bandwidths.")
print("Very narrow bandwidths have wide confidence intervals (few observations).")
print("Very wide bandwidths can introduce bias if the outcome-running variable")
print("relationship is nonlinear far from the cutoff.")

## Key Takeaways

1. **Sharp vs. Fuzzy**: In sharp RDD, crossing the cutoff *determines* treatment. In fuzzy RDD, crossing the cutoff *shifts the probability* of treatment. Sharp RDD is a special case of fuzzy RDD where the first stage jump is exactly 1.

2. **Fuzzy RDD = IV/2SLS**: The cutoff indicator $Z = \mathbf{1}(X \geq c)$ serves as an instrument for actual treatment $D$. The Wald estimator divides the reduced form (ITT) by the first stage.

3. **The first stage must be strong**: If the jump in treatment probability at the cutoff is small (weak instrument), the Wald estimator becomes noisy and unreliable. Always check and report the first stage.

4. **The estimate is a LATE**: Fuzzy RDD identifies the treatment effect for **compliers** at the cutoff: people whose treatment status is actually changed by crossing the cutoff. It does not estimate the effect for always-takers or never-takers.

5. **Bandwidth still matters**: The bias-variance tradeoff from sharp RDD carries over. Check sensitivity to bandwidth choice.

**Connection to Mo & Conn (2018)**: Their study of Teach For America uses exactly this fuzzy RDD framework. The selection score cutoff shifts the probability of being placed as a TFA teacher, but many above-cutoff applicants don't end up teaching, and some below-cutoff applicants find other routes in. The 2SLS estimate recovers the effect of actually having a TFA teacher on student test scores.